# 杭州多步预测 TM 因果图对比 notebook

这个 notebook 聚焦杭州站点网络，在固定模型与 support-cover `L_v` 选取规则下，比较不同预测步长、TM 采样数、随机种子和筛边参数对应的三类因果图。默认优先复用已有 cache，只在缺 cache 或你显式要求时重算。


In [14]:
from __future__ import annotations

import os
import sys
from pathlib import Path

from IPython.display import HTML, display
import pandas as pd

NOTEBOOK_TEST_MODE = os.getenv("YRD_NOTEBOOK_TEST_MODE", "0") == "1"
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "yrd").exists() else NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from yrd.air_search_notebook import find_project_root, run_air_tm_notebook_case

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)


## Experiment Config

这里保留实验侧可调参数。默认 `use_cached_results=True`，因此会优先读取现有 `air_search` 缓存；若对应 cache 不存在，才会自动补算。


In [15]:
EXPERIMENT_CONFIG = {
    "city_en": "hangzhou",
    "horizon": 3 if NOTEBOOK_TEST_MODE else 12,
    "tm_sample_count": 32 if NOTEBOOK_TEST_MODE else 4096,
    "sampling_seed": 0 if NOTEBOOK_TEST_MODE else 42,
    "gamma": 1.0,
    "use_cached_results": True,
    "force_retrain": False,
    "force_recompute_tm": False,
}

EXPERIMENT_CONFIG


{'city_en': 'hangzhou',
 'horizon': 12,
 'tm_sample_count': 4096,
 'sampling_seed': 42,
 'gamma': 1.0,
 'use_cached_results': True,
 'force_retrain': False,
 'force_recompute_tm': False}

## Plot Config

这里控制图的筛边策略。`top_k_edges` 用来只保留最强的若干条边；`min_abs_strength` 可额外去掉绝对强度太小的边；`show_negative_synergy_edges` 决定协同图是否把负边也画出来。


In [16]:
GRAPH_CONFIG = {
    "top_k_edges": 8 if NOTEBOOK_TEST_MODE else 10,
    "min_abs_strength": 0.0,
    "show_negative_synergy_edges": True,
}

GRAPH_CONFIG


{'top_k_edges': 10,
 'min_abs_strength': 0.0,
 'show_negative_synergy_edges': True}

## Run Or Load Results

这一格只做薄调用：根据当前参数读取或补算杭州 TM summary，并同步生成三张筛边后的因果图。


In [17]:
results = run_air_tm_notebook_case(
    root_dir=PROJECT_ROOT,
    city_en=EXPERIMENT_CONFIG["city_en"],
    horizon=EXPERIMENT_CONFIG["horizon"],
    tm_sample_count=EXPERIMENT_CONFIG["tm_sample_count"],
    sampling_seed=EXPERIMENT_CONFIG["sampling_seed"],
    gamma=EXPERIMENT_CONFIG["gamma"],
    top_k_edges=GRAPH_CONFIG["top_k_edges"],
    min_abs_strength=GRAPH_CONFIG["min_abs_strength"],
    show_negative_synergy_edges=GRAPH_CONFIG["show_negative_synergy_edges"],
    force_retrain=EXPERIMENT_CONFIG["force_retrain"],
    force_recompute_tm=EXPERIMENT_CONFIG["force_recompute_tm"],
    use_smoke=NOTEBOOK_TEST_MODE,
)

run_context = results["run_context"]
summary_metrics_df = results["summary_metrics_df"]
profile_variable_df = results["profile_variable_df"]
o3_pairwise_display_df = results["o3_pairwise_display_df"]
pm25_to_o3_display_df = results["pm25_to_o3_display_df"]
synergy_display_df = results["synergy_display_df"]
o3_pairwise_ranked_df = results["o3_pairwise_ranked_df"]
pm25_to_o3_ranked_df = results["pm25_to_o3_ranked_df"]
synergy_ranked_df = results["synergy_ranked_df"]
graph_paths = results["graph_paths"]
final_conclusion_text = results["final_conclusion_text"]

run_context


/Users/yangmingzhe/Desktop/code/github/EISyn/yrd/air_search.py:377: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_payload = torch.load(checkpoint_path, map_locati

{'city_en': 'hangzhou',
 'horizon': 12,
 'tm_sample_count': 4096,
 'sampling_seed': 42,
 'gamma': 1.0,
 'box_mode': 'per_variable',
 'run_tag': 'refine_tm_g1p00_m4096_seed42',
 'used_cached_results': True,
 'results_dir': '/Users/yangmingzhe/Desktop/code/github/EISyn/fig/yrd_air_search/hangzhou/12h/refine_tm_g1p00_m4096_seed42/notebook_top10_min0p0000_neg1'}

## Summary Tables

先看预测与因果摘要，再看 `L_v` 的 support-cover 选取。前者帮助你判断当前参数是否仍保留较好的预测精度与正协同；后者用来核对均匀采样是否落在合理支持域。


In [18]:
print(final_conclusion_text)
summary_metrics_df


Hangzhou 12h uses support-cover L_v with gamma=1.00. The notebook defaults to cached TM results, exposes O3 -> O3, PM2.5 -> O3, and O3 + PM2.5 -> O3 synergy graphs, and lets you compare top-10 edges under different sample_count/seed settings.


,metric,value
0,joint_model overall RMSE,26.005579
1,joint_model overall Corr,0.763602
2,joint_model O3 RMSE,33.052269
3,joint_model O3 Corr,0.713313
4,joint_model PM2.5 RMSE,16.128481
5,joint_model PM2.5 Corr,0.724322
6,persistence overall RMSE,50.564934
7,persistence overall Corr,0.203466
8,persistence O3 RMSE,68.753456
9,persistence O3 Corr,-0.086921


In [19]:
profile_variable_df


,variable,center,train_min,train_max,cover_radius,box_size_Lv,lower_bound,nonnegative_clipped
0,O3,0.000119,-1.119915,8.939430,8.824761,17.649523,-1.140074,True
1,PM2.5,-0.000084,-1.316032,15.117043,15.207382,30.414764,-1.352069,True
2,t2m,0.000344,-3.947592,2.522585,3.903087,7.806174,-31.600544,True
3,d2m,0.000339,-3.932643,1.717390,3.870230,7.740460,-31.353790,True
4,sp,-0.000303,-4.432071,2.997128,3.503364,7.006728,-93.743370,True
5,tp,0.000066,-0.286299,30.140316,30.140196,60.280392,-0.286299,True
6,blh,0.000192,-1.015761,5.951709,5.973622,11.947244,-1.038477,True
7,msdwswrf,0.000147,-0.640079,3.664200,3.668174,7.336349,-0.640077,True
8,u100,0.000252,-6.876710,4.821651,6.801450,13.602900,NaN,False
9,v100,-0.000122,-5.583564,3.373910,5.628579,11.257157,NaN,False


In [20]:
edge_table_top_n = 3
edge_tables = {
    "O3 -> O3": o3_pairwise_ranked_df[["edge_label", "mean"]].head(edge_table_top_n),
    "PM2.5 -> O3": pm25_to_o3_ranked_df[["edge_label", "mean"]].head(edge_table_top_n),
    "O3 + PM2.5 -> O3": synergy_ranked_df[["edge_label", "mean"]].head(edge_table_top_n),
}

edge_tables


{'O3 -> O3':        edge_label      mean
 0  3558A -> 1227A  0.151194
 1  1231A -> 3558A  0.131484
 2  1231A -> 3557A  0.106559,
 'PM2.5 -> O3':        edge_label      mean
 0  1228A -> 3557A  0.056370
 1  1228A -> 1230A  0.049798
 2  1228A -> 1223A  0.042548,
 'O3 + PM2.5 -> O3':        edge_label      mean
 0  1231A -> 3558A  0.001497
 1  1228A -> 1223A  0.001438
 2  1228A -> 1226A  0.001134}

## Graphs

下面三张图共用同一套筛边参数，方便直接比较不同 `horizon / sample_count / seed / top_k` 下的形状变化。


In [21]:
cards = []
for title, key in zip(
    ["O3 -> O3", "PM2.5 -> O3", "O3 + PM2.5 -> O3"],
    ["o3_pairwise", "pm25_to_o3_pairwise", "o3_pm25_synergy"],
):
    cards.append(
        f'''<div style="width:100%;margin:0 0 24px 0"><div style="font-weight:600;font-size:18px;margin:0 0 10px 0">{title}</div><img src="{graph_paths[key]}" style="display:block;width:min(1200px,100%);height:auto;border:1px solid #ddd;border-radius:8px;background:#fff" /></div>'''
    )
display(HTML('<div style="display:block;max-width:1280px">' + ''.join(cards) + '</div>'))


## Single Global L Experiment

这一节保留上面的 `support-cover L_v` 结果不动，并暴露一个可手动调节的单值 `L`。当 `global_box_size_override=None` 时，会自动取 `L=max_v L_v`；如果你填入正数，就会直接用这个手动 `L` 重新生成因果图。


In [27]:
GLOBAL_MAX_EXPERIMENT_CONFIG = {
    **EXPERIMENT_CONFIG,
    "box_mode": "global_max",
    "global_box_size_override": 50,  # 例如改成 40.0；None 表示自动取 max_v L_v
}

GLOBAL_MAX_EXPERIMENT_CONFIG


{'city_en': 'hangzhou',
 'horizon': 12,
 'tm_sample_count': 4096,
 'sampling_seed': 42,
 'gamma': 1.0,
 'use_cached_results': True,
 'force_retrain': False,
 'force_recompute_tm': False,
 'box_mode': 'global_max',
 'global_box_size_override': 50}

In [28]:
global_max_results = run_air_tm_notebook_case(
    root_dir=PROJECT_ROOT,
    city_en=GLOBAL_MAX_EXPERIMENT_CONFIG["city_en"],
    horizon=GLOBAL_MAX_EXPERIMENT_CONFIG["horizon"],
    tm_sample_count=GLOBAL_MAX_EXPERIMENT_CONFIG["tm_sample_count"],
    sampling_seed=GLOBAL_MAX_EXPERIMENT_CONFIG["sampling_seed"],
    gamma=GLOBAL_MAX_EXPERIMENT_CONFIG["gamma"],
    top_k_edges=GRAPH_CONFIG["top_k_edges"],
    min_abs_strength=GRAPH_CONFIG["min_abs_strength"],
    show_negative_synergy_edges=GRAPH_CONFIG["show_negative_synergy_edges"],
    force_retrain=GLOBAL_MAX_EXPERIMENT_CONFIG["force_retrain"],
    force_recompute_tm=GLOBAL_MAX_EXPERIMENT_CONFIG["force_recompute_tm"],
    use_smoke=NOTEBOOK_TEST_MODE,
    box_mode=GLOBAL_MAX_EXPERIMENT_CONFIG["box_mode"],
    global_box_size_override=GLOBAL_MAX_EXPERIMENT_CONFIG["global_box_size_override"],
)

global_max_run_context = global_max_results["run_context"]
global_max_summary_metrics_df = global_max_results["summary_metrics_df"]
global_max_profile_variable_df = global_max_results["profile_variable_df"].copy()
global_max_graph_paths = global_max_results["graph_paths"]
global_max_final_conclusion_text = global_max_results["final_conclusion_text"]

global_max_run_context


{'city_en': 'hangzhou',
 'horizon': 12,
 'tm_sample_count': 4096,
 'sampling_seed': 42,
 'gamma': 1.0,
 'box_mode': 'global_max',
 'global_box_size': 50.0,
 'global_box_size_override': 50.0,
 'run_tag': 'refine_tm_g1p00_m4096_seed42_l50p0000',
 'used_cached_results': False,
 'results_dir': '/Users/yangmingzhe/Desktop/code/github/EISyn/fig/yrd_air_search/hangzhou/12h/refine_tm_g1p00_m4096_seed42_l50p0000/notebook_top10_min0p0000_neg1'}

In [29]:
global_max_l = float(global_max_results["profile"]["global_box_size"])
global_max_profile_variable_df["original_box_size_Lv"] = global_max_profile_variable_df["variable"].map(
    global_max_results["profile"]["original_box_size_by_variable"]
)
global_max_profile_variable_df["shared_box_size_L"] = global_max_l

print(global_max_final_conclusion_text)
global_max_summary_metrics_df


Hangzhou 12h uses scalar L=max_v L_v=50.0000 with gamma=1.00. The notebook defaults to cached TM results, exposes O3 -> O3, PM2.5 -> O3, and O3 + PM2.5 -> O3 synergy graphs, and lets you compare top-10 edges under different sample_count/seed settings.


,metric,value
0,joint_model overall RMSE,26.005579
1,joint_model overall Corr,0.763602
2,joint_model O3 RMSE,33.052269
3,joint_model O3 Corr,0.713313
4,joint_model PM2.5 RMSE,16.128481
5,joint_model PM2.5 Corr,0.724322
6,persistence overall RMSE,50.564934
7,persistence overall Corr,0.203466
8,persistence O3 RMSE,68.753456
9,persistence O3 Corr,-0.086921


In [30]:
global_max_profile_variable_df


,variable,center,train_min,train_max,cover_radius,box_size_Lv,lower_bound,nonnegative_clipped,original_box_size_Lv,shared_box_size_L
0,O3,0.000119,-1.119915,8.939430,8.824761,50.0,-1.140074,True,17.649523,50.0
1,PM2.5,-0.000084,-1.316032,15.117043,15.207382,50.0,-1.352069,True,30.414764,50.0
2,t2m,0.000344,-3.947592,2.522585,3.903087,50.0,-31.600544,True,7.806174,50.0
3,d2m,0.000339,-3.932643,1.717390,3.870230,50.0,-31.353790,True,7.740460,50.0
4,sp,-0.000303,-4.432071,2.997128,3.503364,50.0,-93.743370,True,7.006728,50.0
5,tp,0.000066,-0.286299,30.140316,30.140196,50.0,-0.286299,True,60.280392,50.0
6,blh,0.000192,-1.015761,5.951709,5.973622,50.0,-1.038477,True,11.947244,50.0
7,msdwswrf,0.000147,-0.640079,3.664200,3.668174,50.0,-0.640077,True,7.336349,50.0
8,u100,0.000252,-6.876710,4.821651,6.801450,50.0,NaN,False,13.602900,50.0
9,v100,-0.000122,-5.583564,3.373910,5.628579,50.0,NaN,False,11.257157,50.0


In [31]:
global_max_cards = []
for title, key in zip(
    ["O3 -> O3", "PM2.5 -> O3", "O3 + PM2.5 -> O3"],
    ["o3_pairwise", "pm25_to_o3_pairwise", "o3_pm25_synergy"],
):
    global_max_cards.append(
        f'''<div style="width:100%;margin:0 0 24px 0"><div style="font-weight:600;font-size:18px;margin:0 0 10px 0">{title}</div><img src="{global_max_graph_paths[key]}" style="display:block;width:min(1200px,100%);height:auto;border:1px solid #ddd;border-radius:8px;background:#fff" /></div>'''
    )
display(HTML('<div style="display:block;max-width:1280px">' + ''.join(global_max_cards) + '</div>'))


## Single Global L Interpretation

- 如果 `global_box_size_override=None`，这里的 `shared_box_size_L` 就等于原始 `L_v` 里的最大值。
- 如果你手动填了 `global_box_size_override`，三张图就会统一使用这个指定的单值 `L`。
- `original_box_size_Lv` 仍然保留下来作为对照，方便直接比较“按变量盒宽”与“单个全局盒宽”下的差异。
